# Hugging Face Transformers 微调训练入门

本示例将介绍基于 Transformers 实现模型微调训练的主要流程，包括：
- 数据集下载
- 数据预处理
- 训练超参数配置
- 训练评估指标设置
- 训练器基本介绍
- 实战训练
- 模型保存

## YelpReviewFull 数据集

**Hugging Face 数据集：[ YelpReviewFull ](https://huggingface.co/datasets/yelp_review_full)**

## 下载数据集

In [1]:
from datasets import load_dataset

dataset = load_dataset("yelp_review_full")

In [2]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})

## 预处理数据

下载数据集到本地后，使用 Tokenizer 来处理文本，对于长度不等的输入数据，可以使用填充（padding）和截断（truncation）策略来处理。

Datasets 的 `map` 方法，支持一次性在整个数据集上应用预处理函数。

下面使用填充到最大长度的策略，处理整个数据集：

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [4]:
train_dataset = tokenized_datasets["train"].shuffle(seed=164)
eval_dataset = tokenized_datasets["test"].shuffle(seed=164)

## 微调训练配置

### 加载 BERT 模型

警告通知我们正在丢弃一些权重（`vocab_transform` 和 `vocab_layer_norm` 层），并随机初始化其他一些权重（`pre_classifier` 和 `classifier` 层）。在微调模型情况下是绝对正常的，因为我们正在删除用于预训练模型的掩码语言建模任务的头部，并用一个新的头部替换它，对于这个新头部，我们没有预训练的权重，所以库会警告我们在用它进行推理之前应该对这个模型进行微调，而这正是我们要做的事情。

In [5]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("models/bert-base-uncased-finetune-yelp-round5", num_labels=5)

### 训练超参数（TrainingArguments）

完整配置参数与默认值：https://huggingface.co/docs/transformers/v4.36.1/en/main_classes/trainer#transformers.TrainingArguments

源代码定义：https://github.com/huggingface/transformers/blob/v4.36.1/src/transformers/training_args.py#L161

**最重要配置：模型权重保存路径(output_dir)**

In [6]:
from transformers import TrainingArguments, Trainer

model_dir = "models/bert-base-uncased-finetune-yelp-round6"

training_args = TrainingArguments(output_dir=model_dir,
                                  eval_strategy="steps", 
                                  per_device_train_batch_size=65,
                                  num_train_epochs=2,
                                  logging_steps=1000,
                                  weight_decay=0.01,
                                  warmup_steps=3000)

### 训练过程中的指标评估（Evaluate)

**[Hugging Face Evaluate 库](https://huggingface.co/docs/evaluate/index)** 支持使用一行代码，获得数十种不同领域（自然语言处理、计算机视觉、强化学习等）的评估方法。 当前支持 **完整评估指标：https://huggingface.co/evaluate-metric**

训练器（Trainer）在训练过程中不会自动评估模型性能。因此，我们需要向训练器传递一个函数来计算和报告指标。 

Evaluate库提供了一个简单的准确率函数，您可以使用`evaluate.load`函数加载

In [7]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")


接着，调用 `compute` 函数来计算预测的准确率。

在将预测传递给 compute 函数之前，我们需要将 logits 转换为预测值（**所有Transformers 模型都返回 logits**）。

In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

#### 训练过程指标监控

通常，为了监控训练过程中的评估指标变化，我们可以在`TrainingArguments`指定`evaluation_strategy`参数，以便在 epoch 结束时报告评估指标。

## 开始训练

### 实例化训练器（Trainer）

`kernel version` 版本问题：暂不影响本示例代码运行

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

In [10]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy
1000,0.478300,0.759304,0.694720
2000,0.502800,0.763644,0.687440
3000,0.531900,0.745605,0.691060
4000,0.568700,0.751072,0.687800
5000,0.580600,0.731400,0.687940
6000,0.586000,0.735988,0.689420
7000,0.579400,0.724793,0.688960
8000,0.584700,0.740112,0.689300
9000,0.586700,0.726471,0.693560
10000,0.590600,0.711401,0.692320


TrainOutput(global_step=20000, training_loss=0.5054048858642578, metrics={'train_runtime': 15459.9049, 'train_samples_per_second': 84.088, 'train_steps_per_second': 1.294, 'total_flos': 3.420535852032e+17, 'train_loss': 0.5054048858642578, 'epoch': 2.0})

In [11]:
small_test_dataset = tokenized_datasets["test"].shuffle(seed=64).select(range(100))

In [12]:
trainer.evaluate(small_test_dataset)

{'eval_loss': 0.8554030060768127,
 'eval_accuracy': 0.66,
 'eval_runtime': 0.3602,
 'eval_samples_per_second': 277.605,
 'eval_steps_per_second': 36.089,
 'epoch': 2.0}

### 保存模型和训练状态

- 使用 `trainer.save_model` 方法保存模型，后续可以通过 from_pretrained() 方法重新加载
- 使用 `trainer.save_state` 方法保存训练状态

In [13]:
trainer.save_model(model_dir)

In [14]:
trainer.save_state()